In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/ab_data.csv')

# re-apply cleaning
df = df[
    ((df['group'] == 'control') & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
]
df = df.drop_duplicates(subset='user_id', keep='first')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Clean data shape: {df.shape}")

Clean data shape: (290584, 5)


In [2]:
control = df[df['group'] == 'control']['converted']
treatment = df[df['group'] == 'treatment']['converted']

n_control = len(control)
n_treatment = len(treatment)
conv_control = control.mean()
conv_treatment = treatment.mean()

print(f"Control:   n={n_control:,}, conversion={conv_control*100:.4f}%")
print(f"Treatment: n={n_treatment:,}, conversion={conv_treatment*100:.4f}%")
print(f"Absolute difference: {(conv_treatment - conv_control)*100:.4f}%")
print(f"Relative difference: {((conv_treatment - conv_control)/conv_control)*100:.2f}%")

Control:   n=145,274, conversion=12.0386%
Treatment: n=145,310, conversion=11.8808%
Absolute difference: -0.1578%
Relative difference: -1.31%


In [3]:
from statsmodels.stats.proportion import proportions_ztest

conversions = np.array([treatment.sum(), control.sum()])
nobs = np.array([n_treatment, n_control])

z_stat, p_value = proportions_ztest(conversions, nobs)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Alpha: 0.05")
print()
if p_value < 0.05:
    print("RESULT: Statistically significant — reject null hypothesis")
    print("The difference between pages is REAL")
else:
    print("RESULT: Not statistically significant — fail to reject null hypothesis")
    print("The difference between pages is likely just noise")

Z-statistic: -1.3109
P-value: 0.1899
Alpha: 0.05

RESULT: Not statistically significant — fail to reject null hypothesis
The difference between pages is likely just noise


In [4]:
import statsmodels.api as sm

diff = conv_treatment - conv_control
se = np.sqrt(
    (conv_control * (1 - conv_control) / n_control) +
    (conv_treatment * (1 - conv_treatment) / n_treatment)
)
ci_lower = diff - 1.96 * se
ci_upper = diff + 1.96 * se

print(f"Difference: {diff*100:.4f}%")
print(f"95% Confidence Interval: [{ci_lower*100:.4f}%, {ci_upper*100:.4f}%]")
print()
if ci_lower < 0 < ci_upper:
    print("Zero is inside the confidence interval")
    print("We cannot confidently say the new page is better or worse")
else:
    print("Zero is outside the confidence interval")
    print("We can confidently say there is a real difference")

Difference: -0.1578%
95% Confidence Interval: [-0.3938%, 0.0781%]

Zero is inside the confidence interval
We cannot confidently say the new page is better or worse


In [5]:
import pingouin as pg

cohen_h = 2 * np.arcsin(np.sqrt(conv_treatment)) - 2 * np.arcsin(np.sqrt(conv_control))
print(f"Cohen's h (effect size): {cohen_h:.4f}")
print()
if abs(cohen_h) < 0.2:
    print("Effect size: SMALL — even if significant, practical impact is minimal")
elif abs(cohen_h) < 0.5:
    print("Effect size: MEDIUM")
else:
    print("Effect size: LARGE")

Matplotlib is building the font cache; this may take a moment.


Cohen's h (effect size): -0.0049

Effect size: SMALL — even if significant, practical impact is minimal


In [6]:
results = {
    'n_control': n_control,
    'n_treatment': n_treatment,
    'conv_control': conv_control,
    'conv_treatment': conv_treatment,
    'absolute_diff': conv_treatment - conv_control,
    'relative_diff': (conv_treatment - conv_control) / conv_control,
    'z_statistic': z_stat,
    'p_value': p_value,
    'ci_lower': ci_lower,
    'ci_upper': ci_upper,
    'cohen_h': cohen_h,
    'significant': p_value < 0.05
}

results_df = pd.DataFrame([results])
results_df.to_csv('../data/processed/test_results.csv', index=False)
print("Results saved")
print(results_df.T)

Results saved
                       0
n_control         145274
n_treatment       145310
conv_control    0.120386
conv_treatment  0.118808
absolute_diff  -0.001578
relative_diff   -0.01311
z_statistic    -1.310924
p_value         0.189883
ci_lower       -0.003938
ci_upper        0.000781
cohen_h        -0.004864
significant        False
